# Day 10: NVIDIA Dynamo — Disaggregated Serving
> *Inference Engineering* — Chapter 4.4 | Philip Kiely, Baseten Books 2026

**Layer:** Runtime | **Prerequisite:** Day 07 (vLLM), Day 09 (TRT-LLM)


## Concept Overview

NVIDIA Dynamo is a distributed inference framework that disaggregates prefill and decode into separate, independently scalable worker pools. Prefill (processing the input prompt) is compute-intensive; decode (generating tokens one by one) is memory-bandwidth-bound. By separating them, each can be scaled to its optimal hardware and batch size independently. KV cache transfer between prefill and decode workers is a key challenge — Dynamo uses NIXL (NVIDIA Inference Xfer Library) for high-bandwidth NVLink/RDMA KV migration.


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Prefill vs Decode Characteristics

Understanding why disaggregation helps requires understanding the computational difference between the two phases.


In [ ]:
def compute_prefill_flops(seq_len, d_model, num_layers, num_heads, d_head):
    """Approximate FLOPs for prefill phase."""
    # Attention: 2 * seq^2 * d_model per layer (QKV proj + attn + out proj)
    attn_flops = num_layers * (4 * seq_len * d_model * d_model +  # QKV + O proj
                               2 * seq_len**2 * d_model)           # attn scores + AV
    # FFN: 2 * seq * d_model * d_ff
    ffn_flops = num_layers * 2 * seq_len * d_model * 4 * d_model
    return attn_flops + ffn_flops

def compute_decode_mem_bytes(num_layers, num_heads, d_head, dtype_bytes=2):
    """Bytes to load model weights per decode step."""
    # Weight-loading dominates at batch=1
    d_model = num_heads * d_head
    attn_weights = num_layers * 4 * d_model * d_model * dtype_bytes  # QKV + O
    ffn_weights = num_layers * 3 * d_model * 4 * d_model * dtype_bytes  # SwiGLU
    return attn_weights + ffn_weights

# Llama-3-8B configuration
cfg = {'d_model':4096,'num_layers':32,'num_heads':32,'d_head':128}

print('Prefill vs Decode Computation Profile (Llama-3-8B):')
print(f'{"Seq Len":>10} {"Prefill TFLOPs":>18} {"Time@312TF/s":>15}')
print('-' * 45)
for seq in [128, 512, 2048, 8192]:
    flops = compute_prefill_flops(seq, cfg['d_model'], cfg['num_layers'], cfg['num_heads'], cfg['d_head'])
    t_ms = flops / 312e12 * 1000
    print(f'{seq:>10} {flops/1e12:>18.3f} {t_ms:>15.1f} ms')

print()
decode_bytes = compute_decode_mem_bytes(cfg['num_layers'], cfg['num_heads'], cfg['d_head'])
print(f'Decode memory load per step: {decode_bytes/1e9:.1f} GB (at batch=1)')
print(f'Decode time @2TB/s BW: {decode_bytes/2e12*1000:.1f} ms per token')
print(f'Decode time @273GB/s BW (DGX Spark): {decode_bytes/273e9*1000:.1f} ms per token')


## 2. Disaggregated Architecture

Prefill workers run at high batch size on compute-optimized hardware. Decode workers run at high concurrency on memory-bandwidth-optimized hardware. KV cache is transferred between them after prefill completes.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: coupled serving (traditional)
ax = axes[0]
steps = ['Prefill\n(compute)', 'KV in GPU mem', 'Decode\n(mem-BW)', 'Output']
colors = ['#E74C3C','#3498DB','#2ECC71','#9B59B6']
timings = [30, 5, 60, 5]  # ms
x = 0
for step, color, t in zip(steps, colors, timings):
    ax.barh(0, t, left=x, height=0.5, color=color, edgecolor='black')
    ax.text(x + t/2, 0, f'{step}\n({t}ms)', ha='center', va='center', fontsize=8)
    x += t
ax.set_xlim(0, x*1.05)
ax.set_title('Coupled Serving (single GPU)')
ax.set_yticks([])
ax.set_xlabel('Time (ms)')

# Right: disaggregated serving
ax = axes[1]
# Prefill worker
timings_p = [30, 5]  # prefill + KV transfer
colors_p = ['#E74C3C','#F39C12']
x = 0
for step, color, t in zip(['Prefill','KV Transfer'], colors_p, timings_p):
    ax.barh(0.5, t, left=x, height=0.4, color=color, edgecolor='black')
    ax.text(x+t/2, 0.5, f'{step}\n({t}ms)', ha='center', va='center', fontsize=8)
    x += t
# Decode worker (starts after KV transfer)
x_start = sum(timings_p)
timings_d = [60, 5]
colors_d = ['#2ECC71','#9B59B6']
x = x_start
for step, color, t in zip(['Decode','Output'], colors_d, timings_d):
    ax.barh(-0.5, t, left=x, height=0.4, color=color, edgecolor='black')
    ax.text(x+t/2, -0.5, f'{step}\n({t}ms)', ha='center', va='center', fontsize=8)
    x += t
ax.set_xlim(0, 100)
ax.set_yticks([0.5, -0.5])
ax.set_yticklabels(['Prefill\nWorker','Decode\nWorker'])
ax.set_title('Disaggregated Serving (Dynamo)')
ax.set_xlabel('Time (ms)')
ax.axvline(sum(timings_p), color='black', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('disaggregated_serving.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved disaggregated_serving.png')


## 3. KV Cache Transfer Bandwidth

The KV cache must be transferred from prefill GPU to decode GPU over NVLink or RDMA. Transfer latency adds overhead — the benefit of disaggregation only outweighs this when prompt lengths are substantial.


In [ ]:
def kv_transfer_time(seq_len, num_layers, num_heads, d_head, bandwidth_gb_s, dtype_bytes=2):
    """KV cache size and transfer time."""
    kv_bytes = 2 * num_layers * num_heads * d_head * seq_len * dtype_bytes  # K+V
    transfer_ms = kv_bytes / (bandwidth_gb_s * 1e9) * 1000
    return kv_bytes / 1e6, transfer_ms

bandwidths = {
    'PCIe 4.0':     32,    # GB/s
    'NVLink 4.0':   900,   # GB/s (within node)
    'InfiniBand HDR': 25,  # GB/s (inter-node)
    'RDMA (50GbE)':  6,    # GB/s
}

print(f'{"Bandwidth":>20} {"512-tok KV (ms)":>18} {"4096-tok KV (ms)":>20} {"Go/No-go":>10}')
print('-' * 72)
for bw_name, bw in bandwidths.items():
    _, t512 = kv_transfer_time(512, 32, 32, 128, bw)
    _, t4096 = kv_transfer_time(4096, 32, 32, 128, bw)
    viable = 'YES' if t512 < 5 else 'MARGINAL' if t512 < 20 else 'NO'
    print(f'{bw_name:>20} {t512:>18.2f} {t4096:>20.2f} {viable:>10}')

kv_mb, _ = kv_transfer_time(4096, 32, 32, 128, 900)
print(f'\nKV cache size at seq=4096 (Llama-3-8B, FP16): {kv_mb:.1f} MB')


## Experiments: Try These

1. **Simulate prefill/decode disaggregation**: Model arrival traffic, route to prefill workers, track KV transfer time, measure TTFT and decode throughput separately.
2. **NIXL benchmark**: If you have two DGX Sparks, measure actual NVLink bandwidth with `torch.distributed` NCCL point-to-point. Compare to PCIe baseline.
3. **Scaling law**: For what prompt length does disaggregation break even vs coupled serving given your observed KV transfer bandwidth?


## Key Takeaways

- Prefill is compute-bound (scales with seq²), decode is memory-bandwidth-bound (scales with model size × tokens). They have opposite optimal hardware profiles.
- Disaggregated serving routes prefill and decode to separate worker pools, enabling independent scaling and hardware specialization.
- KV cache transfer is the main overhead of disaggregation — NVLink (900 GB/s) makes it practical; PCIe (32 GB/s) makes it marginal for long prompts.
- NVIDIA Dynamo uses NIXL for high-bandwidth KV migration and provides a router layer that abstracts the disaggregated topology from clients.

## References
- *Inference Engineering* Ch 4.4 — Philip Kiely, Baseten Books 2026
- NVIDIA Dynamo documentation (2025)
- Zhong et al. (2024), "DistServe: Disaggregating Prefill and Decoding for Goodput-Optimized LLM Serving"
